# Electricity Cost Analysis: Data Center Deployment Scenarios

**Notebook 04 in the Curtailed-2-Compute Analysis Workflow**

This notebook analyzes electricity costs for three data center deployment scenarios using CAISO Locational Marginal Price (LMP) data.

## Overview

This analysis compares the annual electricity costs for three different data center deployment scenarios:

- **Scenario A**: Metro Baseline - Traditional data center with no flexibility
- **Scenario B**: Rural Flexible Load - Data center that can shift load to curtailment hours
- **Scenario C**: Rural with Battery Storage - Data center with flexible load and battery energy storage system (BESS)

## Workflow Position

- **Input**: Uses LMP data and curtailment identification methodology from previous notebooks
- **Output**: Provides cost analysis results used in `05_scenario_financial_analysis.ipynb` for comprehensive financial modeling

## Methodology

### Curtailment Identification
Curtailment hours are identified when the congestion component of LMP is negative (`lmp_congestion <= 0`). This indicates periods when renewable energy is being curtailed due to oversupply or transmission constraints.

### Cost Components
1. **Energy Charges**: Based on PG&E B-20 tariff structure (time-of-use rates)
   - **Source**: PG&E Schedule B-20 (Secondary Voltage, Time-of-Use)
   - **How to obtain**: Visit [PG&E Tariff Schedules](https://www.pge.com/tariffs/) and search for "Schedule B-20"
   - **Direct link**: [PG&E Schedule B-20](https://www.pge.com/tariffs/tm2/pdf/ELEC_SCHEDS_B-20.pdf)
   - **Note**: Rates are updated periodically. The rates used in this analysis are from 2024.
   - **Update frequency**: Check PG&E website for current rates if analyzing different time periods
2. **Demand Charges**: Based on peak demand during billing period
   - Same source as energy charges (PG&E Schedule B-20)
3. **Curtailed Energy**: Priced at Locational Marginal Price (LMP)
   - During curtailment hours, flexible data centers pay the market LMP instead of retail TOU rates
   - LMP during curtailment is typically lower than retail rates due to oversupply conditions
   - This represents the actual market-based pricing during curtailment periods
   - Provides moderate savings compared to retail rates, while still reflecting market conditions

### Scenario Details

#### Scenario A: Metro Baseline (Retrofitted, Underutilized)
- **Utilization**: 67% of installed capacity (16.08 MW of 24 MW)
- **Rationale**: Models a retrofitted data center where existing infrastructure is repurposed
  - Existing facilities often operate below nameplate capacity due to:
    - Legacy equipment constraints
    - Inefficient space utilization
    - Gradual migration from old to new infrastructure
  - 67% utilization represents a realistic retrofit scenario
- **Pricing**: Standard TOU rates for all hours (no flexibility)
- **Use case**: Baseline comparison representing current state of many existing data centers

#### Scenario B: Rural Flexible Load (New Deployment, Full Capacity)
- **Utilization**: 100% of installed capacity (24 MW)
- **Rationale**: Models a new deployment designed for flexible load operations
  - New facilities can be designed from the ground up to utilize full capacity
  - Flexible load capabilities enable time-shifting to maximize curtailment utilization
  - Full capacity deployment represents the maximum potential for flexible load
- **Pricing**: 
  - During curtailment: Market LMP (typically lower than retail rates)
  - During non-curtailment: Standard TOU rates
- **Use case**: Demonstrates the value proposition of new flexible data center deployments

**Key Distinction**: The utilization difference (67% vs 100%) reflects the operational reality that:
- Retrofitted facilities (Scenario A) typically operate below capacity
- New flexible deployments (Scenario B) can be designed to utilize full capacity
- This comparison shows both the retrofit baseline and the new deployment potential

#### Scenario C: Rural with Battery Storage
- **Base**: Same as Scenario B (100% utilization, flexible load)
- **Additions**: Battery energy storage system (BESS) for:
  - Energy arbitrage (charge low, discharge high)
  - Ancillary services revenue

## Data Requirements

### Source Data Files Needed

**CAISO LMP Data (CSV files)** - **Not included in repository**, must be downloaded:

1. Visit [CAISO OASIS](http://oasis.caiso.com/)
2. Navigate to "Reports" → "Locational Marginal Price"
3. Select:
   - **Node**: `TH_NP15_GEN-APND` (Northern California pricing node)
   - **Time Period**: Monthly files for desired year (e.g., 2024)
   - **Data Items**: All LMP components (LMP_PRC, LMP_CONG_PRC, LMP_ENE_PRC, LMP_LOSS_PRC, LMP_GHG_PRC)
4. Download and save as: `YYYYMM_LMP.csv` (e.g., `202401_LMP.csv`)
5. Place in: `../data/LMP_Data/` directory

**Expected files for 2024 analysis**:
- `202401_LMP.csv` through `202412_LMP.csv` (12 monthly files)

**File size**: ~3,600-3,700 lines per month (hourly data)

**Note**: The notebook searches for files matching pattern `2024*_LMP.csv` in the specified directory.

## Key Parameters

- **IT Load**: 20 MW
- **PUE**: 1.2 (Power Usage Effectiveness)
- **Installed Capacity**: 24 MW (20 MW × 1.2)
- **Battery**: 20 MW power, 80 MWh energy, 85% efficiency



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
# --- Define Tariff and Battery Parameters ---

# PG&E B-20 Tariff Structure (Secondary Voltage)
# Source: PG&E Schedule B-20 (Time-of-Use, Secondary Voltage)
# How to obtain current rates:
#   1. Visit: https://www.pge.com/tariffs/
#   2. Search for "Schedule B-20" or "B-20"
#   3. Direct link: https://www.pge.com/tariffs/tm2/pdf/ELEC_SCHEDS_B-20.pdf
#   4. Look for "Secondary Voltage" rates and "Time-of-Use" schedule
# Note: Rates are updated periodically by PG&E. These rates are from 2024.
#       Update these values if analyzing different time periods.
b20_rates = {
    'energy_charges': {
        'summer_peak': 0.20832,
        'summer_part_peak': 0.16020,
        'summer_off_peak': 0.12220,
        'winter_peak': 0.17965,
        'winter_off_peak': 0.12189,
        'winter_super_off_peak': 0.04451
    },
    'demand_charges': {
        'summer_peak': 50.19,
        'summer_part_peak': 10.81,
        'summer_max': 43.05,
        'winter_peak': 3.22,
        'winter_max': 43.05
    }
}

# Battery parameters for Scenario C
battery_params = {
    'power_mw': 20,
    'energy_mwh': 80,
    'efficiency': 0.85, 
    # CORRECTED LOGIC: Using a more realistic ancillary rate based on market research
    'ancillary_rate': 7 # $7/kW-yr, based on market analysis
}

In [3]:
# Define a function to get the correct TOU period for a given timestamp
# This function is designed to work with localized Pacific Time.
def get_tou_period(dt):
    hour = dt.hour
    month = dt.month
    
    if month >= 6 and month <= 9:  # Summer
        if hour >= 16 and hour < 21:
            return 'summer_peak'
        elif (hour >= 14 and hour < 16) or (hour >= 21 and hour < 23):
            return 'summer_part_peak'
        else:
            return 'summer_off_peak'
    else:  # Winter
        if hour >= 16 and hour < 21:
            return 'winter_peak'
        elif month in [3, 4, 5] and hour >= 9 and hour < 14:
            return 'winter_super_off_peak'
        else:
            return 'winter_off_peak'

In [4]:
# --- Scenario Functions (Updated) ---

def run_scenario_a(df_data, b20_rates):
    """
    Scenario A: Metro Baseline - Traditional data center with no flexibility.
    
    This scenario models a retrofitted, underutilized data center operating at 67% capacity.
    This represents the baseline case where existing infrastructure is repurposed, which
    typically results in lower utilization due to legacy constraints, inefficient space
    utilization, or gradual migration from old to new infrastructure.
    
    Energy costs are calculated using standard PG&E B-20 time-of-use (TOU) rates for all hours.
    Demand charges are based on the full installed capacity, regardless of actual utilization.
    
    Parameters:
    -----------
    df_data : pandas.DataFrame
        DataFrame with datetime index and columns:
        - 'installed_capacity_mw': Total installed capacity in MW
        - 'tou_period': Time-of-use period for each hour
    b20_rates : dict
        Dictionary containing PG&E B-20 tariff rates
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'total_cost': Total annual electricity cost ($)
        - 'energy_cost': Annual energy charges ($)
        - 'demand_cost': Annual demand charges ($)
    
    Notes:
    ------
    - Utilization factor: 67% (represents underutilized/retrofitted facility)
      This is a realistic assumption for retrofitted data centers where:
      * Legacy equipment may not support full capacity utilization
      * Space constraints limit efficient rack deployment
      * Gradual migration prevents immediate full utilization
      * Existing infrastructure may have inefficiencies
    - Energy cost = Load (kW) × TOU rate ($/kWh) for each hour
    - Demand charge = Installed capacity (kW) × Maximum demand rate ($/kW-year)
    - Demand charges use 'summer_max' rate as the most conservative estimate
    """
    df_data = df_data.copy()
    
    # Scenario A models a retrofitted, underutilized data center operating at 67% capacity
    # This represents a typical scenario where existing infrastructure is repurposed
    underutilized_load_mw = df_data['installed_capacity_mw'] * 0.67
    
    # Calculate energy costs: Load (kW) × TOU rate ($/kWh) for each hour
    # Convert MW to kW by multiplying by 1000
    df_data['energy_cost'] = df_data.apply(
        lambda x: (underutilized_load_mw.iloc[0] * 1000) * b20_rates['energy_charges'][x['tou_period']], 
        axis=1
    )
    total_energy_cost = df_data['energy_cost'].sum()
    
    # Demand charges are based on the full installed capacity, not the utilized load
    # This reflects that demand charges are typically based on peak demand or installed capacity
    # Using 'summer_max' rate as the most conservative (highest) estimate
    demand_cost = df_data['installed_capacity_mw'].iloc[0] * 1000 * b20_rates['demand_charges']['summer_max']
    
    total_annual_cost = total_energy_cost + demand_cost
    
    return {
        'total_cost': total_annual_cost,
        'energy_cost': total_energy_cost,
        'demand_cost': demand_cost
    }

In [5]:
def run_scenario_b(df_data, b20_rates):
    """
    Scenario B: Rural Flexible Load - Data center that can shift load to curtailment hours.
    
    This scenario models a flexible data center that can time-shift its computing load to 
    align with periods of renewable energy curtailment. During curtailment hours, the data 
    center pays the Locational Marginal Price (LMP) instead of retail TOU rates, which is 
    typically lower due to oversupply conditions.
    
    **Key Distinction from Scenario A**: This scenario models a NEW DEPLOYMENT at 100% 
    utilization, representing a facility designed from the ground up for flexible load 
    operations. Unlike Scenario A's retrofitted facility (67% utilization), new deployments 
    can be optimized to utilize full installed capacity for flexible load time-shifting.
    
    Curtailment is identified when the congestion component of LMP is negative, indicating
    periods when renewable energy is being curtailed due to oversupply or transmission constraints.
    
    Parameters:
    -----------
    df_data : pandas.DataFrame
        DataFrame with datetime index and columns:
        - 'installed_capacity_mw': Total installed capacity in MW
        - 'tou_period': Time-of-use period for each hour
        - 'lmp_congestion': Congestion component of LMP ($/MWh)
        - 'total_lmp': Total LMP ($/MWh)
    b20_rates : dict
        Dictionary containing PG&E B-20 tariff rates
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'total_cost': Total annual electricity cost ($)
        - 'savings': Savings compared to Scenario A ($)
        - 'curtailment_hours': Number of hours with curtailment conditions
    
    Notes:
    ------
    - Utilization: 100% (full installed capacity = 24 MW)
      This represents a new deployment designed for flexible load operations, where:
      * Infrastructure is designed from the start for flexible operations
      * Full capacity can be utilized for time-shifting to curtailment periods
      * No legacy constraints limit utilization
      * Operational design enables maximum flexible load potential
      This is distinct from Scenario A's 67% utilization (retrofitted, underutilized)
    
    - Curtailment identification: lmp_congestion <= 0
    - Energy cost during curtailment: Load (MW) × LMP ($/MWh) = $/hour
      (Unit conversion: MW × 1000 = kW, LMP ($/MWh) / 1000 = $/kWh, so kW × $/kWh = $/hour)
      LMP during curtailment is typically lower than retail TOU rates, providing savings.
    - Energy cost during non-curtailment: Standard TOU rates
    - Demand charges: Same as Scenario A (based on full installed capacity)
    
    Academic Justification:
    The 100% utilization assumption for Scenario B is justified by:
    1. New deployments can be designed for optimal flexible load operations
    2. Full capacity utilization maximizes the value proposition of flexible load
    3. This represents the theoretical maximum potential for new flexible data centers
    4. Comparison with Scenario A (67%) shows both retrofit baseline and new deployment potential
    """
    df_data = df_data.copy()
    
    # CORRECTED LOGIC: This scenario models a data center at full installed capacity
    # This represents a new deployment that can utilize the full capacity for flexible load
    full_load_mw = df_data['installed_capacity_mw'].iloc[0]
    
    # Identify curtailment hours: negative congestion component indicates curtailment
    # This occurs when renewable oversupply or transmission constraints cause curtailment
    df_data['is_curtailment'] = df_data['lmp_congestion'] <= 0
    
    # Calculate energy costs:
    # - During curtailment: Pay total LMP (typically lower than retail TOU rates)
    #   This represents the actual market price during curtailment periods, which is
    #   typically lower than retail rates due to oversupply conditions.
    # - During non-curtailment: Pay standard TOU rates
    # 
    # Unit conversion: Load (MW) → kW (×1000), LMP ($/MWh) → $/kWh (÷1000)
    # Cost = Load (kW) × Price ($/kWh) = Load (MW) × 1000 × LMP ($/MWh) / 1000 = Load (MW) × LMP ($/MWh)
    # For hourly cost: Load (MW) × LMP ($/MWh) = $ (since 1 hour × MW = MWh)
    df_data['energy_cost_flex'] = df_data.apply(
        lambda x: (full_load_mw * 1000) * x['total_lmp'] / 1000 if x['is_curtailment']
        else (full_load_mw * 1000) * b20_rates['energy_charges'][x['tou_period']], 
        axis=1
    )
    total_energy_cost = df_data['energy_cost_flex'].sum()
    
    # Demand charges remain the same as Scenario A (based on full capacity)
    demand_cost = df_data['installed_capacity_mw'].iloc[0] * 1000 * b20_rates['demand_charges']['summer_max']
    
    total_annual_cost = total_energy_cost + demand_cost
    
    # Calculate savings compared to Scenario A baseline
    scenario_a_result = run_scenario_a(df_data, b20_rates)
    savings = scenario_a_result['total_cost'] - total_annual_cost
    
    curtailment_hours = df_data['is_curtailment'].sum()
    print(f"Curtailment hours identified: {curtailment_hours} ({curtailment_hours/len(df_data)*100:.1f}% of data)")
    
    return {
        'total_cost': total_annual_cost,
        'savings': savings,
        'curtailment_hours': curtailment_hours
    }

In [6]:
def run_scenario_c(df_data, b20_rates, battery_params):
    """
    Scenario C: Rural with Battery Energy Storage System (BESS) - Flexible load + battery storage.
    
    This scenario adds a Battery Energy Storage System (BESS) to Scenario B, enabling:
    1. Energy arbitrage: Charge during low-price hours, discharge during high-price hours
    2. Ancillary services: Provide grid services (frequency regulation, etc.)
    
    The battery performs daily arbitrage by:
    - Charging during the 4 lowest-price hours of each day
    - Discharging during the 4 highest-price hours of each day
    
    Parameters:
    -----------
    df_data : pandas.DataFrame
        DataFrame with datetime index and columns:
        - 'installed_capacity_mw': Total installed capacity in MW
        - 'tou_period': Time-of-use period for each hour
        - 'lmp_energy', 'lmp_losses', 'lmp_congestion', 'lmp_ghg': LMP components ($/MWh)
        - 'total_lmp': Total LMP ($/MWh)
    b20_rates : dict
        Dictionary containing PG&E B-20 tariff rates
    battery_params : dict
        Dictionary containing:
        - 'power_mw': Battery power rating in MW
        - 'energy_mwh': Battery energy capacity in MWh
        - 'efficiency': Round-trip efficiency (0-1)
        - 'ancillary_rate': Ancillary services rate ($/kW-year)
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'net_cost': Net annual electricity cost after battery revenues ($)
        - 'base_cost': Base cost from Scenario B ($)
        - 'arbitrage_revenue': Annual energy arbitrage revenue ($)
        - 'ancillary_revenue': Annual ancillary services revenue ($)
        - 'total_revenue': Total battery revenue ($)
    
    Notes:
    ------
    - Battery charges during 4 lowest-price hours per day
    - Battery discharges during 4 highest-price hours per day
    - Charge cost: LMP ($/MWh) × Power (MW) × Hours = $ (no unit conversion needed)
    - Discharge revenue: LMP ($/MWh) × Power (MW) × Efficiency × Hours = $
      (accounts for round-trip efficiency losses)
    - Ancillary services revenue is based on power rating (kW) × rate ($/kW-year)
    - Net cost = Scenario B cost - arbitrage revenue - ancillary revenue
    """
    df_data = df_data.copy()
    
    # Calculate daily energy arbitrage revenue
    # Strategy: Charge during lowest-price hours, discharge during highest-price hours
    daily_arbitrage = []
    
    for date, daily_df in df_data.groupby(df_data.index.date):
        # Sort by total LMP to identify cheapest and most expensive hours
        daily_df = daily_df.sort_values('total_lmp')
        
        # Charge during 4 lowest-price hours
        charge_hours = daily_df.head(4)
        # Charge cost = LMP ($/MWh) × Power (MW) × Hours = $ (for each hour)
        # For 4 hours: Sum of (LMP × Power) for each hour
        # Note: Using total_lmp (sum of components) for consistency
        # Unit: $/MWh × MW × 1 hour = $ (no conversion needed)
        charge_cost = (
            charge_hours['lmp_energy'] + 
            charge_hours['lmp_losses'] + 
            charge_hours['lmp_congestion'] + 
            charge_hours['lmp_ghg']
        ).sum() * battery_params['power_mw']
        
        # Discharge during 4 highest-price hours
        discharge_hours = daily_df.tail(4)
        # Discharge revenue accounts for round-trip efficiency
        # Only (efficiency × energy) can be discharged from stored energy
        # Unit: $/MWh × MW × efficiency × 1 hour = $
        discharge_revenue = (
            discharge_hours['lmp_energy'] + 
            discharge_hours['lmp_losses'] + 
            discharge_hours['lmp_congestion'] + 
            discharge_hours['lmp_ghg']
        ).sum() * battery_params['power_mw'] * battery_params['efficiency']
        
        daily_profit = discharge_revenue - charge_cost
        daily_arbitrage.append(daily_profit)
    
    annual_arbitrage_revenue = sum(daily_arbitrage)
    
    # Ancillary services revenue: Based on power rating (kW) × rate ($/kW-year)
    # This represents revenue from providing grid services like frequency regulation
    ancillary_revenue = battery_params['power_mw'] * 1000 * battery_params['ancillary_rate']
    
    # Base cost is from Scenario B (flexible load without battery)
    scenario_b_result = run_scenario_b(df_data, b20_rates)
    base_cost = scenario_b_result['total_cost']
    
    # Net cost after accounting for battery revenues
    net_cost = base_cost - annual_arbitrage_revenue - ancillary_revenue
    
    return {
        'net_cost': net_cost,
        'base_cost': base_cost,
        'arbitrage_revenue': annual_arbitrage_revenue,
        'ancillary_revenue': ancillary_revenue,
        'total_revenue': annual_arbitrage_revenue + ancillary_revenue
    }

In [7]:
# NOTE: The scenario functions are defined above in cells 4, 5, and 6.
# This cell is intentionally left empty to avoid duplicate function definitions.
# If you need to modify the scenario functions, edit cells 4, 5, and 6 above.

In [8]:
def load_and_combine_data(directory='../data/LMP_Data/', node_id='TH_NP15_GEN-APND'):
    """
    Loads and combines all monthly LMP files into a single DataFrame.
    
    Searches for LMP files in the specified directory. If not found, provides
    instructions for downloading the data.
    """
    # Try the specified directory first
    all_files = glob.glob(os.path.join(directory, "2024*_LMP.csv"))
    
    # If not found, try alternative locations
    if not all_files:
        alt_dirs = ['../data/LMP_Data/', '../data/']
        for alt_dir in alt_dirs:
            alt_files = glob.glob(os.path.join(alt_dir, "2024*_LMP.csv"))
            if alt_files:
                all_files = alt_files
                directory = alt_dir
                print(f"Found files in alternative location: {directory}")
                break
    
    if not all_files:
        print("=" * 70)
        print("ERROR: No LMP data files found!")
        print("=" * 70)
        print(f"Searched in: {directory}")
        print("\nTo obtain LMP data:")
        print("1. Visit: http://oasis.caiso.com/")
        print("2. Navigate to: Reports → Locational Marginal Price")
        print("3. Select:")
        print("   - Node: TH_NP15_GEN-APND")
        print("   - Time Period: Monthly files for 2024")
        print("   - Data Items: All LMP components")
        print("4. Download files as: YYYYMM_LMP.csv (e.g., 202401_LMP.csv)")
        print(f"5. Place files in: {directory}")
        print("=" * 70)
        return pd.DataFrame()
        
    df_list = []
    
    for file in all_files:
        print(f"Processing file: {os.path.basename(file)}")
        df = pd.read_csv(file)
        
        df = df[df['NODE_ID'] == node_id].copy()
        
        df = df.pivot(index='INTERVALSTARTTIME_GMT', columns='XML_DATA_ITEM', values='MW')
        df.reset_index(inplace=True)
        
        df_list.append(df)
        
    final_df = pd.concat(df_list, ignore_index=True)
    
    final_df.rename(columns={
        'LMP_PRC': 'total_lmp',
        'LMP_CONG_PRC': 'lmp_congestion',
        'LMP_ENE_PRC': 'lmp_energy',
        'LMP_LOSS_PRC': 'lmp_losses',
        'LMP_GHG_PRC': 'lmp_ghg',
        'INTERVALSTARTTIME_GMT': 'datetime'
    }, inplace=True)
    
    final_df['datetime'] = pd.to_datetime(final_df['datetime'])
    final_df.set_index('datetime', inplace=True)
    
    final_df = final_df.tz_convert('America/Los_Angeles')
    
    # Define the load and PUE for the model
    it_load_mw = 20.0
    pue = 1.2
    final_df['installed_capacity_mw'] = it_load_mw * pue
    
    final_df['tou_period'] = final_df.index.to_series().apply(get_tou_period)

    return final_df[['total_lmp', 'lmp_congestion', 'lmp_energy', 'lmp_losses', 'lmp_ghg', 'installed_capacity_mw', 'tou_period']]

In [9]:
full_year_df = load_and_combine_data()

if full_year_df.empty:
    print("DataFrame is empty. Cannot run scenarios.")
else:
    print("\nFull year of data combined and prepared successfully.")
    
    print("\nRunning Scenario A: Retrofitting Existing Datacenters")
    results_a = run_scenario_a(full_year_df, b20_rates)

    print("\nRunning Scenario B: Rural with Flexible Load")
    results_b = run_scenario_b(full_year_df, b20_rates)

    print("\nRunning Scenario C: Rural with BESS")
    results_c = run_scenario_c(full_year_df, b20_rates, battery_params)

    print("\n--- Final Results ---")
    print(f"Scenario A - Annual Cost: ${results_a['total_cost'] / 1e6:.2f} M")
    print(f"Scenario B - Annual Cost: ${results_b['total_cost'] / 1e6:.2f} M")
    print(f"Scenario C - Net Annual Cost: ${results_c['net_cost'] / 1e6:.2f} M")

    # --- Visualization Code (unchanged, will use the combined data) ---
    plt.figure(figsize=(12, 7))
    hourly_trends = full_year_df[['lmp_energy', 'lmp_congestion', 'total_lmp']].groupby(full_year_df.index.hour).mean()
    plt.plot(hourly_trends.index, hourly_trends['lmp_energy'], label='Energy Component', linestyle='--', color='blue')
    plt.plot(hourly_trends.index, hourly_trends['lmp_congestion'], label='Congestion Component', linestyle='--', color='red')
    plt.plot(hourly_trends.index, hourly_trends['total_lmp'], label='Total LMP', color='green', linewidth=2)
    plt.title('Average Hourly LMP and Components (The Duck Curve)')
    plt.xlabel('Hour of the Day')
    plt.ylabel('Price ($/MWh)')
    plt.xticks(range(24))
    plt.legend()
    plt.grid(True)
    plt.savefig('daily_lmp_trends.png')
    
    # Export Average Hourly LMP and Components
    hourly_trends.to_csv('avg_hourly_lmp_components_duck_curve.csv')
    print("[OK] Exported: avg_hourly_lmp_components_duck_curve.csv")

    plt.figure(figsize=(12, 7))
    weekly_trends = full_year_df[['total_lmp', 'lmp_congestion']].resample('W').mean()
    plt.plot(weekly_trends.index, weekly_trends['total_lmp'], label='Total LMP', color='purple', linewidth=2)
    plt.plot(weekly_trends.index, weekly_trends['lmp_congestion'], label='Congestion Component', color='orange', linestyle='--')
    plt.title('Weekly Average LMP and Congestion Over 2024')
    plt.xlabel('Date')
    plt.ylabel('Price ($/MWh)')
    plt.legend()
    plt.grid(True)
    plt.savefig('seasonal_lmp_trends.png')

ERROR: No LMP data files found!
Searched in: ../data/LMP_Data/

To obtain LMP data:
1. Visit: http://oasis.caiso.com/
2. Navigate to: Reports → Locational Marginal Price
3. Select:
   - Node: TH_NP15_GEN-APND
   - Time Period: Monthly files for 2024
   - Data Items: All LMP components
4. Download files as: YYYYMM_LMP.csv (e.g., 202401_LMP.csv)
5. Place files in: ../data/LMP_Data/
DataFrame is empty. Cannot run scenarios.


In [31]:
# --- New Analysis Function ---
def analyze_curtailed_savings(df_data, b20_rates):
    """
    Analyzes the specific cost of curtailed vs. non-curtailed energy.
    
    This function provides a detailed breakdown consistent with Scenario B's
    modeling assumption: curtailed energy is priced at LMP (market price),
    which is typically lower than retail TOU rates during oversupply periods.
    """
    df = df_data.copy()
    underutilized_load_mw = df['installed_capacity_mw'] * 0.67
    
    # Define curtailed hours based on negative congestion
    df['is_curtailment'] = df['lmp_congestion'] <= 0
    
    # Calculate the cost of energy for curtailed hours
    # Consistent with Scenario B: curtailed energy is priced at LMP
    # This represents the actual market price during curtailment periods
    cost_curtailed_hours = df[df['is_curtailment']].apply(
        lambda x: (underutilized_load_mw.iloc[0] * 1000) * x['total_lmp'] / 1000, 
        axis=1
    ).sum()
    
    # Calculate the cost of energy for non-curtailed hours (using b20 rates)
    cost_non_curtailed_hours = df[~df['is_curtailment']].apply(
        lambda x: (underutilized_load_mw.iloc[0] * 1000) * b20_rates['energy_charges'][x['tou_period']], 
        axis=1
    ).sum()
    
    # The total cost of the flexible load is the sum of these two
    total_flexible_cost = cost_curtailed_hours + cost_non_curtailed_hours
    
    # The baseline cost for comparison is the cost if no curtailed energy was used
    baseline_cost_energy = df.apply(
        lambda x: (underutilized_load_mw.iloc[0] * 1000) * b20_rates['energy_charges'][x['tou_period']], 
        axis=1
    ).sum()
    
    # The savings are the difference between the baseline and the flexible cost
    savings_from_flex = baseline_cost_energy - total_flexible_cost
    
    return {
        'cost_curtailed_hours': cost_curtailed_hours,
        'cost_non_curtailed_hours': cost_non_curtailed_hours,
        'total_flexible_cost': total_flexible_cost,
        'savings': savings_from_flex
    }

# --- Main Execution (Updated) ---

# This code assumes the 'full_year_df' is already created by your load_and_combine_data function
# ...

# Run all three scenarios
results_a = run_scenario_a(full_year_df, b20_rates)
results_b = run_scenario_b(full_year_df, b20_rates)
results_c = run_scenario_c(full_year_df, b20_rates, battery_params)

# Run the new analysis function
curtailed_analysis = analyze_curtailed_savings(full_year_df, b20_rates)

print("\n--- Detailed Financial Summary ---")
print(f"1. Scenario A (Retrofit Baseline Cost): ${results_a['total_cost'] / 1e6:.2f} M")
print(f"2. Savings from Flexible Load (Scenario B vs. A): ${results_b['savings'] / 1e6:.2f} M")
print(f"   -> Resulting in Scenario B Cost: ${results_b['total_cost'] / 1e6:.2f} M")
print(f"3. Total BESS Revenue (Energy Arbitrage + Ancillary): ${results_c['total_revenue'] / 1e6:.2f} M")
print(f"   -> Resulting in Scenario C Net Cost: ${results_c['net_cost'] / 1e6:.2f} M")

print("\n--- Summary of BESS Revenue ---")
print(f"Energy Arbitrage:  ${results_c['arbitrage_revenue'] / 1e6:.2f} M")
print(f"Ancillary Services:  ${results_c['ancillary_revenue'] / 1e6:.2f} M")

print("\n--- Detailed Breakdown of Flexible Load Savings ---")
print(f"Cost of Energy during Curtailed Hours: ${curtailed_analysis['cost_curtailed_hours'] / 1e6:.2f} M")
print(f"Cost of Energy during Non-Curtailed Hours: ${curtailed_analysis['cost_non_curtailed_hours'] / 1e6:.2f} M")
print(f"Total Energy Cost of Flexible Load: ${curtailed_analysis['total_flexible_cost'] / 1e6:.2f} M")
print(f"Total Savings from Flexible Load Energy: ${curtailed_analysis['savings'] / 1e6:.2f} M")

Curtailment hours identified: 4749 (54.1% of data)
Curtailment hours identified: 4749 (54.1% of data)

--- Detailed Financial Summary ---
1. Scenario A (Retrofit Baseline Cost): $19.97 M
2. Savings from Flexible Load (Scenario B vs. A): $1.92 M
   -> Resulting in Scenario B Cost: $18.04 M
3. Total BESS Revenue (Energy Arbitrage + Ancillary): $0.91 M
   -> Resulting in Scenario C Net Cost: $17.14 M

--- Summary of BESS Revenue ---
Energy Arbitrage:  $0.77 M
Ancillary Services:  $0.14 M

--- Detailed Breakdown of Flexible Load Savings ---
Cost of Energy during Curtailed Hours: $3.17 M
Cost of Energy during Non-Curtailed Hours: $8.23 M
Total Energy Cost of Flexible Load: $11.40 M
Total Savings from Flexible Load Energy: $7.54 M
